[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/43_gradient_accumulation.ipynb)

# 🟡 Medium: Gradient Accumulation

*Training*
Accumulate gradients across a list of micro-batches into **one** gradient
pytree, purely functionally, so that a single optimizer step is mathematically
identical to one step on the concatenated batch.

Let $\ell_j$ be the per-example loss and let micro-batch $i$ hold $n_i$
examples. `grad_fn` returns the gradient of the **mean** loss over the batch it
is given:

$$g_i = \nabla_\theta \frac{1}{n_i}\sum_{j \in B_i} \ell_j
\qquad\text{but the full-batch gradient is}\qquad
g^\star = \nabla_\theta \frac{1}{N}\sum_{j} \ell_j,\; N = \sum_i n_i$$

Recovering $g^\star$ therefore means undoing each micro-batch's own denominator:

$$g^\star = \frac{1}{N}\sum_i n_i\, g_i$$

### Rules
- Signature: `accumulate_grads(grad_fn, params, micro_batches)`
- `grad_fn(params, batch) -> (loss, grads)`; `loss` is the **mean** over `batch`
  and `grads` is a pytree matching `params`
- A batch is a pytree whose leaves all share a leading example axis; its size is
  `jax.tree.leaves(batch)[0].shape[0]`. **Sizes may differ between micro-batches.**
- Return `(loss, grads)` — the size-weighted mean loss and the size-weighted
  mean gradient, i.e. exactly what `grad_fn(params, concat(micro_batches))`
  would have returned
- Call `grad_fn` **once per micro-batch**. Concatenating the micro-batches and
  calling it once defeats the entire purpose
- Combine trees with `jax.tree.map`, not hand-written recursion over dict keys
- Raise `ValueError` on an empty `micro_batches`

### The averaging subtlety
Almost every tutorial writes `loss / n_micro_batches` and stops. That is correct
**only when all micro-batches are the same size**, and it fails silently the
moment they are not — which is precisely the common case: the ragged last chunk
of an epoch, a bucketed-by-length dataloader, or a per-host shard that does not
divide evenly.

With sizes $(3, 5, 2)$ the naive $\frac{1}{3}(g_1+g_2+g_3)$ weights each of the
2 examples in the last micro-batch by $\frac{1}{3}\cdot\frac{1}{2} = 0.167$ while
each of the 5 examples in the middle one gets $\frac{1}{3}\cdot\frac{1}{5} =
0.067$. Short micro-batches quietly dominate the update. Nothing crashes, no
shape is wrong, the loss curve still goes down — you just are not optimising the
objective you think you are.

The same bug in language-model training is worse, because there the natural unit
is the **token**, not the sequence. If your loss is a mean over unmasked tokens,
the accumulation weight must be each micro-batch's unmasked-token count, not its
sequence count. Padding-heavy micro-batches otherwise get over-weighted, and the
effective objective drifts with your batch-shuffling seed.

### Why it matters
Accumulation trades compute for memory: activations for one micro-batch at a
time, but the optimizer sees the statistics of the full batch. It is how a large
effective batch size survives on limited HBM, and it is the sequential twin of
data parallelism — the weighted sum here is exactly the weighted all-reduce a
multi-host job performs.

One thing JAX gives you for free: there is no mutable `.grad` buffer, so the
classic "forgot to zero the gradients" bug cannot be written. The accumulator is
an explicit value you create with `jnp.zeros_like` and thread through the loop.
The failure mode moves entirely to the *weighting*, which is the part the
interviewer is actually probing.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def accumulate_grads(grad_fn, params, micro_batches):
    """Accumulate micro-batch gradients into one full-batch-equivalent gradient.

    Args:
        grad_fn:       callable (params, batch) -> (loss, grads); `loss` is the
                       MEAN loss over `batch` and `grads` matches `params`
        params:        parameter pytree
        micro_batches: list of batches. The number of examples in a batch is
                       jax.tree.leaves(batch)[0].shape[0] and may differ
                       between micro-batches.

    Returns:
        (loss, grads) identical to calling `grad_fn` on the concatenation of
        every micro-batch.

    Raises:
        ValueError: if `micro_batches` is empty.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

params = {"w": jnp.array([[1.0], [-2.0], [0.5]]), "b": jnp.array([0.25])}


def loss_fn(p, batch):
    x, y = batch
    return jnp.mean((x @ p["w"] + p["b"] - y) ** 2)


grad_fn = jax.value_and_grad(loss_fn)

key = jax.random.key(0)
X = jax.random.normal(key, (10, 3))
Y = jax.random.normal(jax.random.key(1), (10, 1))

full_loss, full_grads = grad_fn(params, (X, Y))

# Deliberately ragged: 3 + 5 + 2 = 10
chunks = [(X[:3], Y[:3]), (X[3:8], Y[3:8]), (X[8:], Y[8:])]
acc_loss, acc_grads = accumulate_grads(grad_fn, params, chunks)

naive = jax.tree.map(
    lambda *gs: sum(gs) / len(gs), *[grad_fn(params, c)[1] for c in chunks]
)

print("full batch  w-grad:", full_grads["w"].ravel())
print("accumulated w-grad:", acc_grads["w"].ravel(), "  <- matches")
print("naive mean  w-grad:", naive["w"].ravel(), "  <- silently different")
print("loss:", float(full_loss), float(acc_loss))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("gradient_accumulation")

# hint("gradient_accumulation")      # stuck? nudge without the answer
# solution("gradient_accumulation")  # spoiler: the reference implementation